In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a basic General Matrix Multiplication (GEMM). Given matrix $A$ of dimensions $M \times K$, matrix $B$ of dimensions $K \times N$, input/output matrix $C$ of dimensions $M \times N$, and scalar multipliers $ \alpha $ and $ \beta $, compute the operation:
    $$ C = \alpha \cdot (A \times B) + \beta \cdot C_{initial} $$
</p>
<p>
    The input matrices $A$, $B$, and the initial state of $C$ contain 16-bit floating-point numbers (FP16/<code>half</code>). All matrices are stored in row-major order. The scalars $ \alpha $ and $ \beta $ are 32-bit floats.
</p>

<h2>Implementation Requirements</h2>
<ul>
    <li>Use only native features (external libraries other than WMMA are not permitted).</li>
    <li>The <code>solve</code> function signature must remain unchanged.</li>
    <li>Accumulation during multiplication should use FP32 for better precision before converting the final result to FP16.</li>
    <li>The final result must be stored back into matrix <code>C</code> as <code>half</code>.</li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
<em>(Note: Input matrices A, B, C_initial are FP16 type for the problem)</em><br>
Matrix $A$ ($M=2, K=3$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 \\
4.0 & 5.0 & 6.0
\end{bmatrix}
$$
Matrix $B$ ($K=3, N=2$):
$$
\begin{bmatrix}
1.0 & 2.0 \\
3.0 & 4.0 \\
5.0 & 6.0
\end{bmatrix}
$$
Matrix $C_{initial}$ ($M=2, N=2$):
$$
\begin{bmatrix}
1.0 & 1.0 \\
1.0 & 1.0
\end{bmatrix}
$$
$$\alpha = 1.0 \text{ (FP32)}$$
$$\beta = 0.0 \text{ (FP32)}$$

Output (FP16):<br>
Matrix $C$ ($M=2, N=2$):
$$
\begin{bmatrix}
22.0 & 28.0 \\
49.0 & 64.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
    <li>16 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 4096</li>

  <li>Performance is measured with <code>K</code> = 1,024, <code>M</code> = 1,024, <code>N</code> = 1,024</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_fp16.h>
#include <cuda_runtime.h>

// A, B, and C are device pointers
extern "C" void solve(const half* A, const half* B, half* C, int M, int N, int K, float alpha,
                      float beta) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor,
    B: cute.Tensor,
    C: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
    alpha: cute.Float32,
    beta: cute.Float32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(
    A: jax.Array, B: jax.Array, M: int, N: int, K: int, alpha: float, beta: float
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Float16, MutExternalOrigin],
    B: UnsafePointer[Float16, MutExternalOrigin],
    C: UnsafePointer[Float16, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
    alpha: Float32,
    beta: Float32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# A, B, C are tensors on the GPU
def solve(
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    M: int,
    N: int,
    K: int,
    alpha: float,
    beta: float,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# a, b, c are tensors on the GPU
def solve(
    a: torch.Tensor,
    b: torch.Tensor,
    c: torch.Tensor,
    M: int,
    N: int,
    K: int,
    alpha: float,
    beta: float,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="General Matrix Multiplication (GEMM)",
            atol=5e-2,
            rtol=5e-2,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        A: torch.Tensor,
        B: torch.Tensor,
        C: torch.Tensor,
        M: int,
        N: int,
        K: int,
        alpha: float,
        beta: float,
    ):
        assert A.shape == (M, K)
        assert B.shape == (K, N)
        assert C.shape == (M, N)
        A_f32 = A.view(M, K).to(torch.float32)
        B_f32 = B.view(K, N).to(torch.float32)
        C_f32 = C.view(M, N).to(torch.float32)
        matmul_result = torch.matmul(A_f32, B_f32)
        final_result = alpha * matmul_result + beta * C_f32
        C.copy_(final_result.to(torch.float16))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "A": (ctypes.POINTER(ctypes.c_uint16), "in"),
            "B": (ctypes.POINTER(ctypes.c_uint16), "in"),
            "C": (ctypes.POINTER(ctypes.c_uint16), "inout"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
            "K": (ctypes.c_int, "in"),
            "alpha": (ctypes.c_float, "in"),
            "beta": (ctypes.c_float, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float16
        A = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], device="cuda", dtype=dtype)
        B = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], device="cuda", dtype=dtype)
        C = torch.tensor([[1.0, 1.0], [1.0, 1.0]], device="cuda", dtype=dtype)
        return {
            "A": A,
            "B": B,
            "C": C,
            "M": 2,
            "N": 2,
            "K": 3,
            "alpha": 1.0,
            "beta": 0.0,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float16
        tests = []

        # 16x16x16_a1_b0
        tests.append(
            {
                "A": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "B": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.zeros((16, 16), device="cuda", dtype=dtype),
                "M": 16,
                "N": 16,
                "K": 16,
                "alpha": 1.0,
                "beta": 0.0,
            }
        )

        # 16x16x16_a1_b1
        tests.append(
            {
                "A": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-0.5, 0.5),
                "B": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-0.5, 0.5),
                "C": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-0.5, 0.5),
                "M": 16,
                "N": 16,
                "K": 16,
                "alpha": 1.0,
                "beta": 1.0,
            }
        )

        # 32x16x16_a0.5_b0.5
        tests.append(
            {
                "A": torch.empty((32, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "B": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((32, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "M": 32,
                "N": 16,
                "K": 16,
                "alpha": 0.5,
                "beta": 0.5,
            }
        )

        # 16x32x16_a1_b1
        tests.append(
            {
                "A": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "B": torch.empty((16, 32), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((16, 32), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "M": 16,
                "N": 32,
                "K": 16,
                "alpha": 1.0,
                "beta": 1.0,
            }
        )

        # 16x16x32_a0_b1
        tests.append(
            {
                "A": torch.empty((16, 32), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "B": torch.empty((32, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "C": torch.empty((16, 16), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "M": 16,
                "N": 16,
                "K": 32,
                "alpha": 0.0,
                "beta": 1.0,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float16
        M = 1024
        N = 1024
        K = 1024
        A = torch.empty((M, K), device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        B = torch.empty((K, N), device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        C = torch.empty((M, N), device="cuda", dtype=dtype).uniform_(-1.0, 1.0)
        return {
            "A": A,
            "B": B,
            "C": C,
            "M": M,
            "N": N,
            "K": K,
            "alpha": 1.0,
            "beta": 1.0,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
